# Neural Preset · 阶段 0（Kaggle T4）

对应 `REPRODUCTION_PLAN.md` §10。目标只有一个：**流程跑通**，不追求论文效果。

## 跑之前

1. Notebook 右侧 `Settings → Accelerator: GPU T4 x2`（代码只用 `cuda:0`，第二张卡不会用到）；
2. `Settings → Internet`：**开**最省事（torchvision 下 21 MB 的 EfficientNet-B0 权重、`pip install pillow-lut`）。
   关网也能跑：`efficientnet_b0-*.pth` 放进 code dataset，`pretrained.mode: local` + 填 `ckpt`；
   `pillow-lut` 的 wheel 也放进 code dataset，把 cell 2 的安装改成装本地 wheel；
3. 需要 2~3 个 Data 挂载（slug 随意，路径由 cell 1 **自动探测**）：

| 挂载 | 内容 |
|---|---|
| 代码 | 本仓库打成的 `code.zip`（`git archive` 或直接 zip，**不含 data/**）；可顺带放权重与 wheel |
| 图片 | COCO 子集（10k 张左右）。**目录层级随意**，可以有一层 `train2017/` |
| LUT | 308 个 `.cube`（目录层级随意）。默认后端直接读原文，无需预先打包 |

**不需要清单文件**：图片按相对路径排序后取前 10000 张，同样的 dataset 每次顺序完全一样
（`train.py` 会打印名单指纹，方便核对两次跑的是不是同一批图）。

评测另用「训练没见过的尾巴」：排序后第 10001 张起随机配 200 对，cell 5 现场生成。

**首次运行几乎一定要改一两处**（路径 / dtype / shape）。别改 notebook 里的逻辑，改包里的代码，
本地提交后再重新上传 zip —— 这样第二次就不用记忆「上次在哪个 cell 里手改过」。

In [ ]:
# 1) 解压代码 + 环境自检 + 自动探测输入目录
import glob, os, shutil, subprocess, sys
from collections import Counter

IN, WORK = "/kaggle/input", "/kaggle/working"
CODE_DIR = f"{WORK}/code"
SRC_ZIP = f"{IN}/neuralpreset-code/code.zip"

if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)
os.makedirs(CODE_DIR, exist_ok=True)
subprocess.run(["unzip", "-q", "-o", SRC_ZIP, "-d", CODE_DIR], check=True)
sys.path.insert(0, CODE_DIR)
# 注意：sys.path 只改本内核。`!python -m ...` 起的是另一个进程，看不到上面这行；
# 而仓库根没有 pyproject.toml / setup.py，包没法 pip install，只能靠 cwd 或 PYTHONPATH；
# 内核 cwd 是 /kaggle/working，不是 CODE_DIR，所以必须把 CODE_DIR 写进 PYTHONPATH。
os.environ["PYTHONPATH"] = CODE_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

# ---- 自动探测：.cube 所在目录、图片最多的目录 ----
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

cubes = [p for p in sorted(glob.glob(f"{IN}/**/*.cube", recursive=True))
         if not p.startswith(CODE_DIR)]
assert cubes, f"{IN} 下没找到 .cube —— 检查 LUT 那个 dataset 是否挂上了"
CUBES = os.path.dirname(cubes[0])
print(f"\n[自动] LUT: {len(cubes)} 个 .cube，例如 {cubes[:2]}\n        → CUBES = {CUBES}")

counts = Counter()
for dirpath, _, filenames in os.walk(IN):
    n = sum(1 for f in filenames if f.lower().endswith(IMG_EXT))
    if n:
        counts[dirpath] = n
assert counts, f"{IN} 下没找到图片 —— 检查图片那个 dataset 是否挂上了"
print("\n[候选] 图片最多的 5 个目录：")
for d, n in counts.most_common(5):
    print(f"  {n:7d} 张  {d}")
IMAGES = counts.most_common(1)[0][0]
print(f"\n[自动] 图片: {counts[IMAGES]} 张 → IMAGES = {IMAGES}")
print("挑错了就手动改这一行：IMAGES = '/kaggle/input/<你的目录>'")

assert "/neuralpreset-code" not in CUBES, "cube 探测到了代码 dataset 里，手动指定 CUBES"
print("\n如果 LUT 目录里混了别的数据集，手动改 CUBES。")

In [ ]:
# 2) 装 pillow-lut（默认 LUT 后端）+ 核对数据规模
!{sys.executable} -m pip install -q pillow-lut

from neuralpreset.data.coco import scan_images
import pillow_lut  # noqa: F401  装没装上，import 一下就知道

names = scan_images(IMAGES)          # 递归 + 排序，与训练时用的顺序完全一致
N_TRAIN = 10000
print(f"共 {len(names)} 张，前 3 张：{names[:3]}")
assert len(names) > N_TRAIN + 50, \
    f"只有 {len(names)} 张，留不出评测用的尾巴；把 N_TRAIN 调小（比如 {max(100, len(names) - 300)}）"
print(f"训练取前 {N_TRAIN} 张；评测用第 {N_TRAIN + 1} 张起的 {len(names) - N_TRAIN} 张（训练没见过的）")

# 可选后端（LUT 涨到几千条、409 MB 内存吃不下时才需要）
# !{sys.executable} {CODE_DIR}/scripts/prepare_lut.py --cube-dir {CUBES} --out-npy {WORK}/lut33_u8.npy

In [ ]:
# 3) 冒烟：300 张 × 2 epoch，约 1 分钟，能暴露 90% 的路径/形状错误
CFG = f"{CODE_DIR}/configs/phase0.yaml"
COMMON = f"--config {CFG} --data-root {IMAGES} --data-list none --cube-dir {CUBES}"

!{sys.executable} -m neuralpreset.train {COMMON} --smoke --out {WORK}/out_smoke --limit 300

In [ ]:
# 4) 全量：10k 张 × 8 epoch。中断后回来重跑这一格（会接上 out/ 里最新的 ckpt）
!{sys.executable} -m neuralpreset.train {COMMON} --out {WORK}/out

In [ ]:
# 5) 出图 + 指标（A4/A5）：用训练没见过的尾巴，现场生成 pairs.txt
import random

CKPT = f"{WORK}/out/last.pt"
val = names[N_TRAIN:]                 # cell 2 已经扫好了，顺序和训练一致
rng = random.Random(0)
pairs, seen = [], set()
while len(pairs) < 200:
    i, j = rng.randrange(len(val)), rng.randrange(len(val))
    if i == j or (i, j) in seen:
        continue
    seen.add((i, j))
    pairs.append((val[i], val[j]))
PAIRS = f"{WORK}/pairs.txt"
with open(PAIRS, "w", encoding="utf-8") as fh:
    for c, s in pairs:
        fh.write(f"{c} {s}\n")
print(f"{PAIRS}: {len(pairs)} 对，例如 {pairs[0]}")

!{sys.executable} -m neuralpreset.infer --ckpt {CKPT} --content {IMAGES}/{pairs[0][0]} --style {IMAGES}/{pairs[0][1]} --out {WORK}/demo.png --matrix --compare
!{sys.executable} -m neuralpreset.evaluate --ckpt {CKPT} --content-dir {IMAGES} --style-dir {IMAGES} --pairs {PAIRS} --n-pairs 200 --out {WORK}/eval --patch 512

In [ ]:
# 6) 归档（A7）：/kaggle/working 里的文件才是能下载的产物
import glob, os, shutil

os.makedirs(f"{WORK}/out", exist_ok=True)
for p in glob.glob(f"{WORK}/demo*.png") + glob.glob(f"{WORK}/eval/metrics.*"):
    shutil.copy2(p, f"{WORK}/out/")

archive = shutil.make_archive(f"{WORK}/neuralpreset_stage0", "zip", WORK, "out")
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")

log = f"{WORK}/out/metrics.jsonl"
if os.path.isfile(log):
    print(open(log, encoding="utf-8").read().strip().split("\n")[-1])